In [2]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

%conda install -c conda-forge numpy pandas matplotlib scikit-learn opencv -y

Looking in indexes: https://download.pytorch.org/whl/cu126
Note: you may need to restart the kernel to use updated packages.




==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c conda-forge conda




3 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: win-64
Solving environment: done

## Package Plan ##

  environment location: c:\Users\dangi\Desktop\Riboflavin-Learning\.conda

  added / updated specs:
    - matplotlib
    - numpy
    - opencv
    - pandas
    - scikit-learn


The following NEW packages will be INSTALLED:

  _libavif_api       conda-forge/win-64::_libavif_api-1.4.1-h57928b3_0 
  _openmp_mutex      conda-forge/win-64::_openmp_mutex-4.5-20_gnu 
  aom                conda-forge/win-64::aom-3.9.1-he0c23c2_0 
  aws-c-auth         conda-forge/win-64::aws-c-auth-0.10.1-h8b39d88_3 
  aws-c-cal          conda-forge/win-64::aws-c-cal-0.9.13-h46f3b43_1 
  aws-c-common       conda-forge/win-64::aws-c-common-0.12.6-hfd05255_0 
  aws-c-compression  conda-forge/win-64::aws-c-compression-0.3.2-hcb3a2da_0 
  aws-c-http         conda-forge/win-64::aws-c-http-0.10.13-h612f3e8_0 
  aws-c-io           conda-forge/win-64::aws-c-io-0.26.3-h0d5b9f9_1 
  aws-



==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c conda-forge conda




In [1]:
import torch
# This forces the GPU to initialize immediately
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

: 

In [2]:
import torch
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, ConcatDataset

# 1. Standard Transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

base_path = 'Data/'
# Get the paths for Riboflavin_04_09, Riboflavin_04_15, etc.
date_folders = [os.path.join(base_path, d) for d in os.listdir(base_path) 
                if os.path.isdir(os.path.join(base_path, d))]

all_sub_datasets = []

for folder in date_folders:
    # This looks at 0.75, 1.5, etc. INSIDE each date folder
    sub_ds = datasets.ImageFolder(root=folder, transform=transform)
    all_sub_datasets.append(sub_ds)
    print(f"Loaded {folder}: {len(sub_ds)} images.")

# Combine them!
full_dataset = ConcatDataset(all_sub_datasets)

# Important: To get the class names, we look at the first sub-dataset
class_names = all_sub_datasets[0].classes
num_classes = len(class_names)

# Split 80/20
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_set, test_set = torch.utils.data.random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False)

print(f"\nTOTAL: {len(full_dataset)} images across {num_classes} concentration classes.")
print(f"Classes: {class_names}")

ModuleNotFoundError: No module named 'torch'

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# 1. Load Pre-trained ResNet18
# Using the updated weights parameter
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 2. Freeze the 'backbone'
# This speeds up training and prevents overfitting on your 4.4k images
for param in model.parameters():
    param.requires_grad = False

# 3. Replace the Final Layer
# We change the output features to match your num_classes
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes)

# 4. Move to your 5070 Ti
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 5. Define Loss Function and Optimizer
# CrossEntropy is perfect for multi-class concentration labels
criterion = nn.CrossEntropyLoss()
# We only train the weights of the final layer (model.fc)
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

print(f"Model initialized for {num_classes} classes and moved to {device}.")

In [ ]:
import time

epochs = 10
print(f"Starting Training...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    running_corrects = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        # Backward pass + Optimize
        loss.backward()
        optimizer.step()

        # Stats
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(train_set)
    epoch_acc = running_corrects.double() / len(train_set)

    print(f'Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}')

print("\nTRAINING COMPLETE!")

# Save the weights so you don't have to retrain later
torch.save(model.state_dict(), 'riboflavin_weights.pth')